# Autoresearch on Google Colab

This notebook runs [Karpathy's autoresearch](https://github.com/karpathy/autoresearch) on Google Colab with a GPU runtime.

**What it does:** An AI agent autonomously modifies `train.py` to improve an LLM's validation loss. Each experiment trains for 5 minutes, checks if the result improved, keeps or discards the change, and repeats — indefinitely.

**Requirements:** A Colab runtime with an NVIDIA GPU (T4 works, A100/H100 is better).

---

## 1. Check GPU & Setup Environment

In [ ]:
# Check that we have a GPU
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU.")

In [ ]:
# Install dependencies
# Note: Colab already has PyTorch with CUDA. We install the additional packages.
!pip install -q kernels rustbpe tiktoken pyarrow

## 2. Download Data & Train Tokenizer

This downloads a few training shards from Karpathy's climbmix dataset and trains a BPE tokenizer. Takes ~2 minutes.

In [ ]:
# Download data and train tokenizer (one-time setup)
# Use --num-shards to control how many training shards to download.
# Default is 10 which is plenty for autoresearch experiments.
!python prepare.py --num-shards 10

## 3. Verify: Run a Single Training Experiment

Before starting the autonomous agent loop, verify that training works on your GPU. This runs a single 5-minute training pass.

In [ ]:
# Run a single training experiment to verify everything works.
# This takes ~5 minutes (plus startup/compilation time).
# If you get OOM errors, reduce DEVICE_BATCH_SIZE in train.py (see section 4 below).
!python train.py

## 4. GPU-Specific Tuning (if needed)

Karpathy's defaults are tuned for an H100 (80GB VRAM). If you're on a smaller GPU (T4 = 16GB, L4 = 24GB, A100 = 40/80GB), you may need to adjust settings.

**If the training run above crashed with OOM**, uncomment and run the cell below to auto-patch `train.py` for your GPU.

Alternatively, edit `train.py` directly — the hyperparameters section near the bottom is clearly labeled.

In [ ]:
# === UNCOMMENT AND RUN THIS CELL IF YOU GOT OOM ERRORS ===
# This patches train.py with conservative settings for smaller GPUs.
# Adjust values based on your GPU (check nvidia-smi output above).

# import re
#
# gpu_name = torch.cuda.get_device_name(0).lower()
# vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
#
# # Conservative defaults by VRAM tier
# if vram_gb < 20:  # T4 (16GB)
#     depth, batch_size, total_batch, window = 4, 32, 2**16, '"L"'
# elif vram_gb < 30:  # L4 (24GB)
#     depth, batch_size, total_batch, window = 6, 64, 2**17, '"SL"'
# elif vram_gb < 50:  # A100 40GB
#     depth, batch_size, total_batch, window = 8, 64, 2**18, '"SSSL"'
# else:  # A100 80GB / H100
#     depth, batch_size, total_batch, window = 8, 128, 2**19, '"SSSL"'
#
# with open('train.py', 'r') as f:
#     code = f.read()
#
# code = re.sub(r'^DEPTH = \d+', f'DEPTH = {depth}', code, flags=re.MULTILINE)
# code = re.sub(r'^DEVICE_BATCH_SIZE = \d+', f'DEVICE_BATCH_SIZE = {batch_size}', code, flags=re.MULTILINE)
# code = re.sub(r'^TOTAL_BATCH_SIZE = .*', f'TOTAL_BATCH_SIZE = {total_batch}', code, flags=re.MULTILINE)
# code = re.sub(r'^WINDOW_PATTERN = .*', f'WINDOW_PATTERN = {window}', code, flags=re.MULTILINE)
#
# with open('train.py', 'w') as f:
#     f.write(code)
#
# print(f"Patched train.py for {gpu_name} ({vram_gb:.0f}GB VRAM):")
# print(f"  DEPTH = {depth}")
# print(f"  DEVICE_BATCH_SIZE = {batch_size}")
# print(f"  TOTAL_BATCH_SIZE = {total_batch}")
# print(f"  WINDOW_PATTERN = {window}")
# print("\nRe-run the training cell above to verify it works now.")

## 5. Initialize Git for the Agent Loop

The autoresearch agent uses git to track experiments. Each improvement gets committed; failed experiments get reverted.

In [ ]:
import subprocess, datetime

# Initialize git repo if not already
subprocess.run(["git", "init"], check=True)
subprocess.run(["git", "config", "user.email", "autoresearch@colab"], check=True)
subprocess.run(["git", "config", "user.name", "autoresearch"], check=True)

# Commit current state as baseline
subprocess.run(["git", "add", "-A"], check=True)
subprocess.run(["git", "commit", "-m", "baseline", "--allow-empty"], check=True)

# Create autoresearch branch
tag = datetime.datetime.now().strftime("%b%d").lower()
branch = f"autoresearch/{tag}"
subprocess.run(["git", "checkout", "-b", branch], check=True)
print(f"\nReady on branch: {branch}")

## 6. Run the Autonomous Agent Loop

This is the core of autoresearch: a Python loop that acts as the autonomous agent. It:
1. Proposes a modification to `train.py`
2. Runs a 5-minute training experiment
3. Checks if val_bpb improved
4. Keeps (commits) or discards (reverts) the change
5. Repeats forever

**The recommended way** is to point a real LLM agent (Claude Code, Cursor, Codex, etc.) at `program.md` in this repo. The agent reads the instructions and runs the loop itself.

**But if you want a self-contained version** that runs right here in Colab without an external agent, the cell below implements a simple automated hyperparameter search loop using the same keep/discard pattern.

In [ ]:
# ============================================================
# Self-contained autoresearch loop (no external LLM agent needed)
# ============================================================
# This implements the core autoresearch pattern: propose a change,
# train for 5 minutes, keep if improved, discard if not, repeat.
#
# It randomly samples hyperparameter modifications from a search
# space. For smarter exploration (reading code, planning experiments,
# combining ideas), use a real LLM agent with program.md instead.
# ============================================================

import subprocess
import re
import random
import time
import csv
import os

# --- Configuration ---
MAX_EXPERIMENTS = 50  # Set to None for infinite loop
TIMEOUT_SECONDS = 720  # Kill a run if it exceeds 12 minutes

# --- Hyperparameter search space ---
# Each entry: (parameter_name, regex_pattern, candidate_values, description_template)
SEARCH_SPACE = [
    ("DEPTH", r"^DEPTH = \d+", [4, 6, 8, 10, 12], "depth={}"),
    ("ASPECT_RATIO", r"^ASPECT_RATIO = \d+", [48, 56, 64, 72, 80], "aspect_ratio={}"),
    ("EMBEDDING_LR", r"^EMBEDDING_LR = [\d.]+", [0.3, 0.4, 0.6, 0.8, 1.0], "embedding_lr={}"),
    ("UNEMBEDDING_LR", r"^UNEMBEDDING_LR = [\d.]+", [0.002, 0.004, 0.006, 0.008], "unembedding_lr={}"),
    ("MATRIX_LR", r"^MATRIX_LR = [\d.]+", [0.02, 0.03, 0.04, 0.05, 0.06], "matrix_lr={}"),
    ("WEIGHT_DECAY", r"^WEIGHT_DECAY = [\d.]+", [0.05, 0.1, 0.2, 0.3, 0.4], "weight_decay={}"),
    ("WARMDOWN_RATIO", r"^WARMDOWN_RATIO = [\d.]+", [0.3, 0.4, 0.5, 0.6, 0.7], "warmdown_ratio={}"),
    ("WARMUP_RATIO", r"^WARMUP_RATIO = [\d.]+", [0.0, 0.02, 0.05, 0.1], "warmup_ratio={}"),
    ("WINDOW_PATTERN", r'^WINDOW_PATTERN = "[A-Z]+"', ['"L"', '"SL"', '"SSL"', '"SSSL"', '"SSSSL"'], "window={}"),
]


def read_train_py():
    with open("train.py", "r") as f:
        return f.read()


def write_train_py(code):
    with open("train.py", "w") as f:
        f.write(code)


def run_experiment():
    """Run train.py and return (val_bpb, peak_vram_mb) or (None, None) on crash."""
    try:
        result = subprocess.run(
            ["python", "train.py"],
            capture_output=True, text=True,
            timeout=TIMEOUT_SECONDS
        )
        output = result.stdout + result.stderr

        # Write log for debugging
        with open("run.log", "w") as f:
            f.write(output)

        # Parse results
        bpb_match = re.search(r"^val_bpb:\s+([\d.]+)", output, re.MULTILINE)
        vram_match = re.search(r"^peak_vram_mb:\s+([\d.]+)", output, re.MULTILINE)

        if bpb_match:
            val_bpb = float(bpb_match.group(1))
            peak_vram = float(vram_match.group(1)) if vram_match else 0.0
            return val_bpb, peak_vram
        else:
            print(f"  [CRASH] No val_bpb found. Last 20 lines of output:")
            for line in output.strip().split("\n")[-20:]:
                print(f"    {line}")
            return None, None

    except subprocess.TimeoutExpired:
        print(f"  [TIMEOUT] Experiment exceeded {TIMEOUT_SECONDS}s")
        return None, None
    except Exception as e:
        print(f"  [ERROR] {e}")
        return None, None


def git_commit(message):
    subprocess.run(["git", "add", "train.py"], check=True)
    subprocess.run(["git", "commit", "-m", message], check=True, capture_output=True)
    result = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    return result.stdout.strip()


def git_revert():
    subprocess.run(["git", "checkout", "--", "train.py"], check=True)


def log_result(commit, val_bpb, memory_gb, status, description):
    """Append a row to results.tsv."""
    header_needed = not os.path.exists("results.tsv")
    with open("results.tsv", "a") as f:
        if header_needed:
            f.write("commit\tval_bpb\tmemory_gb\tstatus\tdescription\n")
        f.write(f"{commit}\t{val_bpb:.6f}\t{memory_gb:.1f}\t{status}\t{description}\n")


def propose_change(code):
    """Randomly pick a hyperparameter and propose a new value."""
    param_name, pattern, candidates, desc_template = random.choice(SEARCH_SPACE)

    # Find current value
    match = re.search(pattern, code, re.MULTILINE)
    if not match:
        return code, f"skip (couldn't find {param_name})"

    current_line = match.group(0)

    # Pick a new value (different from current if possible)
    new_val = random.choice(candidates)
    new_line = f"{param_name} = {new_val}"

    if new_line == current_line:
        # Try again with a different value
        other_candidates = [c for c in candidates if f"{param_name} = {c}" != current_line]
        if not other_candidates:
            return code, "skip (no alternatives)"
        new_val = random.choice(other_candidates)
        new_line = f"{param_name} = {new_val}"

    new_code = re.sub(pattern, new_line, code, count=1, flags=re.MULTILINE)
    description = desc_template.format(new_val)
    return new_code, description


# ============================================================
# Main loop
# ============================================================

print("=" * 60)
print("AUTORESEARCH LOOP")
print("=" * 60)

# Step 1: Run baseline
print("\n[Experiment 0] Running baseline...")
best_bpb, baseline_vram = run_experiment()
if best_bpb is None:
    raise RuntimeError("Baseline training failed! Fix train.py first (see section 4 above).")

baseline_commit = git_commit("baseline")
log_result(baseline_commit, best_bpb, baseline_vram / 1024, "keep", "baseline")
print(f"  Baseline: val_bpb={best_bpb:.6f}, VRAM={baseline_vram:.0f}MB")
print(f"  Commit: {baseline_commit}")

# Step 2: Experiment loop
experiment_num = 1
n_kept = 0
n_discarded = 0
n_crashed = 0

while MAX_EXPERIMENTS is None or experiment_num <= MAX_EXPERIMENTS:
    print(f"\n{'=' * 60}")
    print(f"[Experiment {experiment_num}] (best so far: {best_bpb:.6f} | kept: {n_kept} | discarded: {n_discarded} | crashed: {n_crashed})")

    # Propose a change
    original_code = read_train_py()
    new_code, description = propose_change(original_code)

    if description.startswith("skip"):
        print(f"  Skipping: {description}")
        continue

    print(f"  Trying: {description}")

    # Apply change
    write_train_py(new_code)
    commit_hash = git_commit(description)

    # Run experiment
    t0 = time.time()
    val_bpb, peak_vram = run_experiment()
    elapsed = time.time() - t0

    if val_bpb is None:
        # Crash — revert
        print(f"  CRASHED after {elapsed:.0f}s — reverting")
        git_revert()
        subprocess.run(["git", "reset", "HEAD~1", "--hard"], check=True, capture_output=True)
        log_result(commit_hash, 0.0, 0.0, "crash", description)
        n_crashed += 1
    elif val_bpb < best_bpb:
        # Improvement — keep!
        improvement = best_bpb - val_bpb
        print(f"  KEEP! val_bpb={val_bpb:.6f} (improved by {improvement:.6f}) in {elapsed:.0f}s")
        log_result(commit_hash, val_bpb, peak_vram / 1024, "keep", description)
        best_bpb = val_bpb
        n_kept += 1
    else:
        # No improvement — discard
        print(f"  DISCARD: val_bpb={val_bpb:.6f} (no improvement, best={best_bpb:.6f}) in {elapsed:.0f}s")
        git_revert()
        subprocess.run(["git", "reset", "HEAD~1", "--hard"], check=True, capture_output=True)
        log_result(commit_hash, val_bpb, peak_vram / 1024, "discard", description)
        n_discarded += 1

    experiment_num += 1

print(f"\n{'=' * 60}")
print(f"DONE! Ran {experiment_num - 1} experiments.")
print(f"Best val_bpb: {best_bpb:.6f}")
print(f"Kept: {n_kept} | Discarded: {n_discarded} | Crashed: {n_crashed}")
print(f"Results saved to results.tsv")

## 7. Analyze Results

After the loop finishes (or you interrupt it), visualize the experiment progress.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Outcomes: {df['status'].value_counts().to_dict()}")
print()

# Show kept experiments
kept = df[df["status"] == "KEEP"]
print("Kept improvements:")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  bpb={row['val_bpb']:.6f}  {row['description']}")

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
valid = df[df["status"] != "CRASH"].reset_index(drop=True)

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bpb"], c="#cccccc", s=12, alpha=0.5, label="Discarded")

kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bpb"], c="#2ecc71", s=50, zorder=4,
           label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum
kept_mask = valid["status"] == "KEEP"
if kept_mask.sum() > 0:
    kept_idx = valid.index[kept_mask]
    running_min = valid.loc[kept_mask, "val_bpb"].cummin()
    ax.step(kept_idx, running_min, where="post", color="#27ae60",
            linewidth=2, alpha=0.7, label="Running best")

ax.set_xlabel("Experiment #")
ax.set_ylabel("Validation BPB (lower is better)")
ax.set_title(f"Autoresearch Progress: {len(df)} Experiments")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("progress.png", dpi=150)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'results.tsv'

## Alternative: Use an LLM Agent

The self-contained loop above does random hyperparameter search. For *smarter* exploration (the way Karpathy intended), point an LLM agent at `program.md`.

Options:
- **Claude Code** (`claude` CLI): `claude` then paste "Hi have a look at program.md and let's kick off a new experiment!"
- **OpenAI Codex**: Point it at this repo
- **Cursor / Windsurf**: Open this folder and reference program.md

The agent reads `program.md`, understands the experiment protocol, and runs the loop — but with the ability to actually read and reason about the code, propose architectural changes, and learn from failed experiments.